# 02 — Modeling: Funding-to-Vulnerability Ratio, Regression & Misallocation Scores

**Stage:** Week 2 of the capstone pipeline.

This notebook takes the cleaned country-year panel from `01_clean.ipynb` and:

1. Computes a **funding-to-vulnerability ratio** for each country-year.
2. Builds and evaluates an **OLS regression** of adaptation aid on vulnerability, governance, income, and geography.
3. Extracts the **regression residuals** as the formal *misallocation score* — the gap between the aid a country actually received and the aid its characteristics predict.

Ranking and residual pattern analysis happen in `03_residual_analysis.ipynb`.

In [30]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

PROCESSED = Path("../data/processed")

panel = pd.read_csv(PROCESSED / "panel.csv")

print(f"Shape: {panel.shape}")
print(f"Countries: {panel['iso3'].nunique()} | Years: {panel['year'].min()}–{panel['year'].max()}\n")
print("Columns:", list(panel.columns), "\n")
panel.head()

Shape: (1883, 11)
Countries: 143 | Years: 2010–2023

Columns: ['iso3', 'country', 'year', 'adaptation_aid_usd_m', 'vulnerability', 'gain_score', 'gdp_per_capita_usd', 'gov_effectiveness', 'income_group', 'is_landlocked', 'is_sids'] 



,iso3,country,year,adaptation_aid_usd_m,vulnerability,gain_score,gdp_per_capita_usd,gov_effectiveness,income_group,is_landlocked,is_sids
0,AFG,Afghanistan,2010,59.470952,0.583800,31.893603,560.621505,20.507295,L,1,0
1,AFG,Afghanistan,2011,129.225633,0.590539,31.075993,606.694676,20.145782,L,1,0
2,AFG,Afghanistan,2012,157.054749,0.584483,32.144508,651.417134,22.178464,L,1,0
3,AFG,Afghanistan,2013,88.860020,0.589310,31.984845,637.087099,22.155125,L,1,0
4,AFG,Afghanistan,2014,93.733749,0.587004,32.918922,625.054942,21.696063,L,1,0


## Step 1 — Funding-to-Vulnerability Ratio

A simple first-pass measure of whether aid tracks need: how many dollars of adaptation
aid a country receives per unit of climate vulnerability.

**Need measure.** The `vulnerability` column (ND-GAIN vulnerability sub-score, 0–1, higher = more
vulnerable) is already a direct "greater value = greater need" measure, so we use it as the
denominator. This is cleaner and more interpretable than inverting the composite GAIN index,
which mixes vulnerability with readiness.

**Ratio:** `aid ÷ vulnerability`
- **High ratio** → well-funded relative to need
- **Low ratio** → underfunded relative to need

This is a descriptive ratio — it controls for *nothing else*. The regression in Step 2 is what
isolates underfunding after accounting for income, governance, and geography. We compute the ratio
first to establish the raw picture the regression will later refine.

In [31]:
# --- Funding-to-vulnerability ratio (per country-year) ---
panel["funding_vuln_ratio"] = panel["adaptation_aid_usd_m"] / panel["vulnerability"]

print("Ratio distribution (aid USD M per unit vulnerability):")
print(panel["funding_vuln_ratio"].describe().round(2), "\n")

# --- Country-level average ratio across all years ---
country_ratio = (
    panel.groupby("country")["funding_vuln_ratio"]
    .mean()
    .sort_values()
)

print("10 LOWEST average ratio (most underfunded, descriptive):")
print(country_ratio.head(10).round(2).to_string(), "\n")

print("10 HIGHEST average ratio (best funded relative to need):")
print(country_ratio.tail(10).round(2).to_string())

Ratio distribution (aid USD M per unit vulnerability):
count    1838.00
mean      196.00
std       371.93
min         0.00
25%        11.01
50%        62.38
75%       215.39
max      4544.22
Name: funding_vuln_ratio, dtype: float64 

10 LOWEST average ratio (most underfunded, descriptive):
country
Oman                                      0.03
Croatia                                   0.06
Barbados                                  0.08
Trinidad and Tobago                       0.29
Equatorial Guinea                         0.65
Turkmenistan                              1.15
Uruguay                                   1.63
Korea, Democratic People's Republic of    2.47
Grenada                                   3.16
Antigua and Barbuda                       5.20 

10 HIGHEST average ratio (best funded relative to need):
country
Colombia                  792.86
Viet Nam                  793.14
Philippines               916.94
Bangladesh               1086.56
Indonesia                1113.23

### Finding: the raw ratio is confounded by income and eligibility

The "most underfunded" countries by raw ratio (Oman, Croatia, Barbados, Trinidad and Tobago,
Uruguay…) are predominantly high- and upper-middle-income, low-vulnerability states. Their low
ratio reflects low *eligibility and need* for adaptation ODA, not misallocation.

This confirms the descriptive ratio cannot isolate misallocation: it conflates "receives little
aid because wealthy/low-risk" with "receives little aid despite high need." The regression in
Step 2 — which controls for GDP per capita, governance, and geography — is what separates the two.
The ratio remains useful as a transparent baseline and a sanity-check column.

## Step 2 — Predictor Completeness Audit (gate before regression)

The OLS uses complete cases across all predictors. Before fitting, we check *how many* rows and
*which countries* fall out, so the modeling sample is a deliberate choice rather than a silent
side effect — and so we can judge whether dropping them biases the sample toward better-documented
countries (which would undercut the hypothesis).

In [32]:
model_cols = ["adaptation_aid_usd_m", "vulnerability", "gain_score",
              "gdp_per_capita_usd", "gov_effectiveness"]

print("Missing per modeling column:")
print(panel[model_cols].isna().sum(), "\n")

complete = panel.dropna(subset=model_cols)
print(f"Full panel:     {len(panel)} rows, {panel['iso3'].nunique()} countries")
print(f"Complete cases: {len(complete)} rows, {complete['iso3'].nunique()} countries")
print(f"Lost:           {len(panel) - len(complete)} rows\n")

dropped = set(panel["iso3"]) - set(complete["iso3"])
print(f"Countries dropped entirely ({len(dropped)}):")
print(sorted(panel[panel["iso3"].isin(dropped)]["country"].dropna().unique()), "\n")

year_loss = (panel.groupby("country").size() - complete.groupby("country").size())
year_loss = year_loss[year_loss > 0].sort_values(ascending=False)
print("Countries losing some (not all) years:")
print(year_loss.head(12).to_string())

Missing per modeling column:
adaptation_aid_usd_m     0
vulnerability           45
gain_score              45
gdp_per_capita_usd      54
gov_effectiveness       14
dtype: int64 

Full panel:     1883 rows, 143 countries
Complete cases: 1806 rows, 138 countries
Lost:           77 rows

Countries dropped entirely (5):
["Korea, Democratic People's Republic of", 'Kosovo', 'Palestine, State of', 'Saint Kitts and Nevis', 'South Sudan'] 

Countries losing some (not all) years:
country
Eritrea                 12.0
Yemen                    5.0
Cuba                     3.0
Syrian Arab Republic     1.0


### Modeling-sample decision

Missingness is concentrated in the most vulnerable countries (informative missingness), not random:
- **5 countries dropped entirely** for missing ND-GAIN vulnerability: DPRK, Kosovo, Palestine,
  Saint Kitts & Nevis, South Sudan. Vulnerability is the core construct and cannot be imputed,
  so these are unrecoverable.
- **Year-coverage reduced** for Eritrea (−12), Yemen (−5), Cuba (−3), Syria (−1), mostly from
  GDP-per-capita gaps.

**Primary model uses listwise deletion: n = 1,806 country-years, 138 countries (4% of rows lost).**
South Sudan is a substantively important omission and is flagged as a limitation, distinct from the
contested-statehood cases (Kosovo, Palestine).

## Step 3 — Freeze the modeling sample (listwise deletion)

**Procedure.** The regression's analytic sample is defined as *complete cases* across the five
modeling variables: any country-year missing the dependent variable or any predictor is dropped.
This yields **n = 1,806 country-years across 138 countries**. The full `panel` is retained for
descriptive work (e.g. the funding-to-vulnerability ratio); the regression and the residual-based
misallocation score are computed only on the frozen `model_df`.

**Rationale for listwise deletion over imputation or interpolation.** Alternative GDP sources were
evaluated before this choice was made; the evidence favored listwise deletion on five grounds:

1. **Limited loss.** Roughly 4% of rows are removed. The remaining 1,806 observations provide ~300
   per predictor (6 predictors), well above the ≥10–20 rule of thumb, so the model remains
   statistically robust.
2. **The core construct is not imputable.** Five countries (DPRK, Kosovo, Palestine, Saint Kitts &
   Nevis, South Sudan) lack an ND-GAIN vulnerability score — the variable the entire question is
   built around. A vulnerability score cannot be fabricated, so these countries are unrecoverable
   under any method, imputation included.
3. **The recoverable data is low quality.** GDP-per-capita gaps are concentrated in
   exchange-rate-distorted economies. The UN figure for Cuba jumps +37.8% in a single year
   (≈ $18,000 against a constant-dollar value of ≈ $7,400); Syria swings ±50% year to year on
   currency collapse. Filling these would inject measurement error into a control variable.
4. **The one clean candidate carries its own bias.** Eritrea has full UN coverage, but its fixed
   15-Nakfa/USD peg overstates dollar GDP. Since GDP per capita enters as a control, an inflated
   value makes Eritrea appear richer, lowering its predicted aid and shrinking its residual — which
   would **mask** the underfunding the analysis aims to detect. Imputing it risks biasing the
   quantity of interest.
5. **Defensibility.** A single-source (World Bank), complete-case model carries no imputation
   assumptions or source-splicing. The exclusions are documented as an explicit limitation, and the
   missingness pattern is itself a finding.

**Trade-off.** Listwise deletion removes the fragile-state tail (South Sudan, Eritrea, Yemen…),
biasing the sample toward better-measured countries. Misallocation estimates are therefore likely
*conservative* for the worst-off, whose true funding gap may exceed what the model can capture. An
optional robustness check is deferred as a stretch task: re-fitting with Eritrea UN-filled to
confirm the conclusions are stable.

In [33]:
# --- Freeze the modeling sample: listwise deletion on the modeling variables ---
model_cols = ["adaptation_aid_usd_m", "vulnerability", "gain_score",
              "gdp_per_capita_usd", "gov_effectiveness"]

# `panel` stays full (descriptive work); `model_df` is the regression frame.
model_df = panel.dropna(subset=model_cols).copy()

n_rows, n_countries = len(model_df), model_df["iso3"].nunique()
print(f"Modeling sample frozen: {n_rows} country-years, {n_countries} countries")
print(f"Dropped {len(panel) - n_rows} of {len(panel)} rows "
      f"({(len(panel) - n_rows) / len(panel):.1%}).")
print(f"Observations per predictor (6 predictors): ~{n_rows // 6}")

# Guard: lock the audited sample so any accidental upstream change is caught early
assert n_rows == 1806,     f"Expected 1806 rows, got {n_rows} — re-check upstream cleaning."
assert n_countries == 138, f"Expected 138 countries, got {n_countries}."
print("\n✓ Sample matches the audited complete-case set.")

Modeling sample frozen: 1806 country-years, 138 countries
Dropped 77 of 1883 rows (4.1%).
Observations per predictor (6 predictors): ~301

✓ Sample matches the audited complete-case set.


## Step 4 — Variable preparation and multicollinearity diagnostics

Two issues must be addressed before fitting OLS:

**1. Skew → log transforms.** Adaptation aid and GDP per capita are both heavily right-skewed
(a few large values dominate), which violates OLS's assumptions about residuals and lets outliers
drive the fit. Both are log-transformed:
- **Aid (dependent variable):** `log1p(aid)` = log(1 + aid). The `1p` form keeps country-years with
  zero or near-zero aid in the sample — those are potentially the *most* underfunded observations and
  must not be discarded. Log of aid flows is also the conventional specification in the
  aid-allocation literature.
- **GDP per capita (predictor):** plain `log`, since the variable is strictly positive.

Modeling aid in log space also gives the residuals a proportional interpretation: a negative residual
means a country received proportionally *less* than its characteristics predict.

**2. Multicollinearity.** Two predictors overlap by construction: the ND-GAIN composite (`gain_score`)
is itself derived partly from vulnerability, so `gain_score` and `vulnerability` are expected to be
strongly correlated. When predictors are collinear, OLS coefficients become unstable and p-values
unreliable — the model cannot attribute effect to one variable versus the other. A correlation matrix
and Variance Inflation Factors (VIF) are computed to quantify this. Interpretation:
- **VIF < 5:** acceptable
- **VIF 5–10:** moderate concern
- **VIF > 10:** serious — one of the collinear predictors should be dropped

The final predictor set is decided from this output, not assumed in advance.

In [34]:
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- Log-transform the skewed variables ---
model_df["log_aid"] = np.log1p(model_df["adaptation_aid_usd_m"])  # log1p keeps aid==0 rows
model_df["log_gdp"] = np.log(model_df["gdp_per_capita_usd"])

print("Skew before -> after log:")
for raw, logged in [("adaptation_aid_usd_m", "log_aid"), ("gdp_per_capita_usd", "log_gdp")]:
    print(f"  {raw:>22}: {model_df[raw].skew():6.2f}  ->  {logged}: {model_df[logged].skew():6.2f}")

# --- Candidate predictor set (per the plan) ---
predictors = ["vulnerability", "gain_score", "log_gdp",
              "gov_effectiveness", "is_landlocked", "is_sids"]

print("\nPredictor correlation matrix:")
print(model_df[predictors].corr().round(2).to_string())

# --- Variance Inflation Factors (constant added for a correct computation; ignore const's own VIF) ---
X = sm.add_constant(model_df[predictors])
vif = pd.DataFrame({
    "predictor": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
print("\nVariance Inflation Factors:")
print(vif.round(2).to_string(index=False))

Skew before -> after log:
    adaptation_aid_usd_m:   5.35  ->  log_aid:  -0.14
      gdp_per_capita_usd:   1.32  ->  log_gdp:  -0.30

Predictor correlation matrix:
                   vulnerability  gain_score  log_gdp  gov_effectiveness  is_landlocked  is_sids
vulnerability               1.00       -0.82    -0.65              -0.46           0.01     0.23
gain_score                 -0.82        1.00     0.75               0.69          -0.11     0.05
log_gdp                    -0.65        0.75     1.00               0.64          -0.29     0.24
gov_effectiveness          -0.46        0.69     0.64               1.00          -0.15     0.15
is_landlocked               0.01       -0.11    -0.29              -0.15           1.00    -0.30
is_sids                     0.23        0.05     0.24               0.15          -0.30     1.00

Variance Inflation Factors:
        predictor    VIF
            const 779.79
    vulnerability   4.75
       gain_score   5.76
          log_gdp   3.28
go

## Step 5 — Extended OLS regression

The model regresses log adaptation aid on vulnerability and the structural controls, testing whether
aid tracks need once income, governance, and geography are held constant.

**Specification:**

`log_aid ~ vulnerability + log_gdp + gov_effectiveness + is_landlocked + is_sids`

Two estimation choices follow from the panel structure:

- **Pooled OLS, not fixed effects.** Country fixed effects would absorb the time-invariant geography
  flags (`is_landlocked`, `is_sids`), making their coefficients inestimable. Since those flags are
  part of the hypothesis, the model is pooled across all country-years.
- **Cluster-robust standard errors by country.** Each country contributes up to 14 yearly
  observations, which are not independent. Naive OLS standard errors would therefore overstate
  precision. Standard errors are clustered on country (`iso3`) so the reported p-values reflect the
  true number of independent units.

`gain_score` is excluded (Step 4): it is collinear with `vulnerability` and redundant with the
development controls.

**Reading the output:** the coefficient on `vulnerability` is the central test — a positive,
significant coefficient would indicate aid rises with need after controls; a null or negative one
would support the misallocation hypothesis. The control coefficients (GDP, governance, geography)
identify which structural factors independently predict the level of aid a country receives.

In [35]:
import statsmodels.formula.api as smf

# Final specification (gain_score dropped for collinearity; vulnerability retained as need measure).
# Pooled OLS with SEs clustered by country to respect the panel's repeated observations.
formula = "log_aid ~ vulnerability + log_gdp + gov_effectiveness + is_landlocked + is_sids"

ols = smf.ols(formula, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["iso3"]}
)

print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                log_aid   R-squared:                       0.259
Model:                            OLS   Adj. R-squared:                  0.257
Method:                 Least Squares   F-statistic:                     22.15
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           3.21e-16
Time:                        23:03:25   Log-Likelihood:                -3341.8
No. Observations:                1806   AIC:                             6696.
Df Residuals:                    1800   BIC:                             6729.
Df Model:                           5                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             7.9443      1.82

## Step 6 — Add a size control (population) and re-fit

The absolute-aid dependent variable conflates country *size* with *prioritization*: small economies
receive small dollar totals regardless of how generously they are treated. The large negative SIDS
coefficient in Step 5 is therefore suspect, and the same distortion would carry into the residual-based
misallocation scores.

To separate size from prioritization, `log(population)` is added as a control. Population is sourced
from the World Bank (`SP.POP.TOTL`), consistent with the GDP-per-capita series, and merged onto the
modeling frame by country and year. Unlike GDP, population is universally reported with no
exchange-rate distortion, so the only validation required is that the merge preserves the sample
(1,806 rows, no missing values).

With population held constant, the coefficients on `is_sids` and `is_landlocked` can be read as
genuine prioritization effects rather than artifacts of country size.

In [36]:
import requests

# --- Source population (World Bank SP.POP.TOTL), same source family as GDP per capita ---
url = ("https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL"
       "?format=json&date=2010:2023&per_page=20000")
records = requests.get(url, timeout=60).json()[1]   # [0] = metadata, [1] = data records

pop = pd.DataFrame(
    [{"iso3": r["countryiso3code"], "year": int(r["date"]), "population": r["value"]}
     for r in records if r["countryiso3code"]]       # skip blank-ISO regional aggregates
).dropna(subset=["population"])

print(f"Pulled population: {pop['iso3'].nunique()} economies, "
      f"{pop['year'].min()}–{pop['year'].max()}, {len(pop)} rows.")

# --- Merge onto the modeling frame and build the size control ---
before = len(model_df)
model_df = model_df.merge(pop, on=["iso3", "year"], how="left")
model_df["log_population"] = np.log(model_df["population"])

missing_pop = int(model_df["population"].isna().sum())
print(f"Rows after merge: {len(model_df)} (was {before}) | missing population: {missing_pop}")

assert len(model_df) == before, "Merge changed row count — duplicate keys in population data."
print("\n" + model_df[["population", "log_population"]].describe().round(2).to_string())

Pulled population: 261 economies, 2010–2023, 3654 rows.
Rows after merge: 1806 (was 1806) | missing population: 0

         population  log_population
count  1.806000e+03         1806.00
mean   4.642585e+07           15.70
std    1.717387e+08            2.23
min    9.816000e+03            9.19
25%    2.108916e+06           14.56
50%    9.781347e+06           16.10
75%    2.968402e+07           17.21
max    1.438070e+09           21.09


In [37]:
# --- Re-fit with the size control added ---
formula = ("log_aid ~ vulnerability + log_gdp + gov_effectiveness "
           "+ log_population + is_landlocked + is_sids")

ols = smf.ols(formula, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["iso3"]}
)
print(ols.summary())

                            OLS Regression Results                            
Dep. Variable:                log_aid   R-squared:                       0.398
Model:                            OLS   Adj. R-squared:                  0.396
Method:                 Least Squares   F-statistic:                     54.59
Date:                Thu, 04 Jun 2026   Prob (F-statistic):           5.80e-34
Time:                        23:03:26   Log-Likelihood:                -3154.3
No. Observations:                1806   AIC:                             6323.
Df Residuals:                    1799   BIC:                             6361.
Df Model:                           6                                         
Covariance Type:              cluster                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept            -3.1183      1.95

## Step 7 — Add year fixed effects (final specification)

Adaptation finance grew sharply over 2010–2023, so a model pooled across years would judge early
and late observations against a single average — mechanically labeling early-year country-years as
underfunded and late-year ones as overfunded purely because the global pool was smaller then. Year
fixed effects (`C(year)`) absorb the size of each year's pool, so the residual measures how a
country fared *relative to comparable countries in the same year*. This is the appropriate
definition of misallocation, and it aligns with the tertiary hypothesis, which asks whether the
allocation *pattern* persisted despite the growth in total aid.

Country fixed effects are deliberately avoided: they would absorb the time-invariant geography flags
and the between-country variation that identifies the vulnerability effect.

The year dummies are nuisance controls and are not interpreted individually. The substantive
coefficients are read as before; the key check is that `vulnerability` remains positive and
significant once a within-year comparison is enforced.

**Income/eligibility tier.** The model so far treats income as a smooth, continuous effect
(`log_gdp`), but adaptation ODA is gated by a *threshold*: upper-middle- and high-income countries
receive little regardless of size or need, because they are not the intended recipients (some are
donors themselves). A continuous income term cannot capture that cliff, so populous middle-income
countries (e.g. China) were being mislabeled as severely "underfunded." Adding `income_group` as a
categorical control (reference = Low income) captures the eligibility tier directly, so the
misallocation score becomes a country's deviation *within its own income/eligibility tier* rather
than against an unrealistic universal baseline.

In [40]:
# Nauru: missing income_group 2010–2014 (leading gap), classified in later years.
# Income tier is slow-moving → back-fill the nearest known classification within country.
model_df = model_df.sort_values(["iso3", "year"])
model_df["income_group"] = model_df.groupby("iso3")["income_group"].bfill().ffill()

print("Missing income_group now:", int(model_df["income_group"].isna().sum()))
print("\nNauru after fill:")
print(model_df[model_df.country == "Nauru"][["year", "income_group", "gdp_per_capita_usd"]]
      .round(0).to_string(index=False))

Missing income_group now: 0

Nauru after fill:
 year income_group  gdp_per_capita_usd
 2010            H              4736.0
 2011            H              6444.0
 2012            H              9843.0
 2013            H              8975.0
 2014            H              9230.0
 2015            H              7747.0
 2016           UM              8748.0
 2017           UM              9657.0
 2018           UM             11414.0
 2019            H             10802.0
 2020            H             10709.0
 2021            H             14979.0
 2022            H             12912.0
 2023            H             12752.0


In [41]:
# The new control was not part of the listwise audit — confirm it's complete on the modeling frame
n_missing_ig = int(model_df["income_group"].isna().sum())
print(f"Missing income_group: {n_missing_ig}")
assert n_missing_ig == 0, "income_group has gaps — handle before adding it as a control."
print("income_group categories:", sorted(model_df["income_group"].unique()), "\n")

# --- FINAL model: add income/eligibility tier (reference = Low income) + year fixed effects ---
formula_fe = ("log_aid ~ vulnerability + log_gdp + gov_effectiveness + log_population "
              "+ is_landlocked + is_sids "
              "+ C(income_group, Treatment(reference='L')) + C(year)")

ols_fe = smf.ols(formula_fe, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["iso3"]}
)

# Compact view: hide only the year dummies (income-tier dummies are shown)
mask = ~ols_fe.params.index.str.contains(r"C\(year\)")
compact = pd.DataFrame({
    "coef": ols_fe.params, "std_err": ols_fe.bse, "p_value": ols_fe.pvalues
})[mask].round(3)
print(f"R² = {ols_fe.rsquared:.3f} | Adj R² = {ols_fe.rsquared_adj:.3f} | n = {int(ols_fe.nobs)}\n")
print(compact.to_string())

Missing income_group: 0
income_group categories: ['H', 'L', 'LM', 'UM'] 

R² = 0.501 | Adj R² = 0.495 | n = 1806

                                                  coef  std_err  p_value
Intercept                                       -4.218    2.031    0.038
C(income_group, Treatment(reference='L'))[T.H]  -0.630    0.584    0.281
C(income_group, Treatment(reference='L'))[T.LM]  0.358    0.239    0.134
C(income_group, Treatment(reference='L'))[T.UM] -0.036    0.379    0.924
vulnerability                                    3.644    1.343    0.007
log_gdp                                         -0.411    0.179    0.022
gov_effectiveness                                0.033    0.008    0.000
log_population                                   0.420    0.049    0.000
is_landlocked                                   -0.142    0.213    0.506
is_sids                                         -0.067    0.264    0.799


## Step 8 — Evaluate and extract misallocation scores

The misallocation score is the residual from the final model: the gap between the log aid a
country-year actually received and the amount predicted from its vulnerability, income, governance,
size, and year. A **negative** residual means the country received *less* than comparable countries
did that year (underfunded); a **positive** residual means *more* (overfunded). Because the model is
in log space, the score is proportional rather than dollar-denominated.

Model quality is assessed two ways:
- **In-sample** R² and mean absolute error (MAE), in log-aid units.
- **Five-fold cross-validation grouped by country**, so no country appears in both training and test
  folds. This tests whether the relationship generalizes to countries the model has not seen — a
  stricter, more honest standard than ordinary k-fold, which would leak a country's other years into
  training. A cross-validated R² somewhat below the in-sample figure is expected and quantifies how
  much of the fit is country-specific.

The scored frame is saved for notebook 03 (ranking and residual pattern analysis).

In [42]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, GroupKFold
from sklearn.metrics import mean_absolute_error

# --- Misallocation score = residual from the final (year-FE) model ---
# negative = underfunded vs same-year peers; positive = overfunded
model_df["predicted_log_aid"]   = ols_fe.fittedvalues
model_df["misallocation_score"] = ols_fe.resid

# --- In-sample fit ---
mae = mean_absolute_error(model_df["log_aid"], model_df["predicted_log_aid"])
print(f"In-sample:  R² = {ols_fe.rsquared:.3f} | MAE (log-aid units) = {mae:.3f}")

# --- 5-fold CV grouped by country (year one-hot so it mirrors the fitted model) ---
X = pd.get_dummies(
        model_df[["vulnerability", "log_gdp", "gov_effectiveness", "log_population",
                  "is_landlocked", "is_sids", "income_group", "year"]],
        columns=["income_group", "year"], drop_first=True).astype(float)
y, groups = model_df["log_aid"], model_df["iso3"]
gkf = GroupKFold(n_splits=5)
cv_r2  =  cross_val_score(LinearRegression(), X, y, groups=groups, cv=gkf, scoring="r2")
cv_mae = -cross_val_score(LinearRegression(), X, y, groups=groups, cv=gkf,
                          scoring="neg_mean_absolute_error")
print(f"5-fold CV (grouped by country):  R² = {cv_r2.mean():.3f} ± {cv_r2.std():.3f} | "
      f"MAE = {cv_mae.mean():.3f} ± {cv_mae.std():.3f}")

# --- Sanity peek (full ranking happens in notebook 03) ---
cols = ["country", "year", "adaptation_aid_usd_m", "vulnerability", "misallocation_score"]
print("\nMost UNDERfunded country-years:")
print(model_df.nsmallest(8, "misallocation_score")[cols].round(2).to_string(index=False))
print("\nMost OVERfunded country-years:")
print(model_df.nlargest(8, "misallocation_score")[cols].round(2).to_string(index=False))

# --- Save scored frame for notebook 03 ---
out = PROCESSED / "model_scored.csv"
model_df.to_csv(out, index=False)
print(f"\nSaved → {out}  ({model_df.shape[0]} rows × {model_df.shape[1]} cols)")

In-sample:  R² = 0.501 | MAE (log-aid units) = 1.023
5-fold CV (grouped by country):  R² = 0.440 ± 0.093 | MAE = 1.068 ± 0.074

Most UNDERfunded country-years:
                  country  year  adaptation_aid_usd_m  vulnerability  misallocation_score
                    China  2023                  8.37           0.38                -3.85
     Syrian Arab Republic  2013                  0.19           0.46                -3.42
                   Gambia  2015                  0.03           0.54                -3.39
Iran, Islamic Republic of  2023                  1.60           0.38                -3.39
     Syrian Arab Republic  2016                  0.78           0.48                -3.32
               Azerbaijan  2019                  0.01           0.40                -3.28
                 Malaysia  2018                  0.64           0.38                -3.19
                   Gambia  2014                  0.01           0.54                -3.14

Most OVERfunded country-years